# KronLLM — одна сессия на Colab (T4)

Ноутбук делает то же, что Kaggle-кернел, но на бесплатной карте Colab:
скачивает корпус, чекпоинт и код с Kaggle, доезжает от последнего шага,
тренирует `--minutes` минут и заливает `last.pt` обратно новой версией
датасета. Следующая сессия (здесь или на Kaggle) продолжит с этого места.

**Что нужно один раз:** Kaggle-токен в *Colab Secrets* (иконка ключа слева):
`KAGGLE_USERNAME` и `KAGGLE_KEY` — те же значения, что в `kaggle.json`.
Токен даёт и чтение датасетов, и запись чекпоинта; больше ничего не нужно.

**Бюджет.** Бесплатный Colab обрывает сессию на 4 часах без предупреждения,
поэтому по умолчанию `MINUTES = 200`: остаётся запас на скачивание и загрузку.
Для Colab Pro поставь 660.

**Важно:** пока вкладка открыта — обучение идёт; закрыл вкладку — сессия
умерла. Чекпоинт пишется каждые 1000 шагов и на выходе по бюджету, так что
потеря ограничена.

Запуск: `Runtime → Change runtime type → T4 GPU`, затем `Runtime → Run all`.

In [ ]:
import shutil, subprocess, sys
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'no GPU')
print('disk free', shutil.disk_usage('/content').free // 2**30, 'GiB')

In [ ]:
import json, os, pathlib

def kaggle_auth():
    """Token from Colab Secrets, or a kaggle.json upload as a fallback."""
    dst = pathlib.Path('/root/.kaggle/kaggle.json')
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import userdata
        user, key = userdata.get('KAGGLE_USERNAME'), userdata.get('KAGGLE_KEY')
    except Exception:
        user = key = None
    if not (user and key):
        from google.colab import files
        print('Secrets KAGGLE_USERNAME / KAGGLE_KEY not set; upload kaggle.json')
        up = files.upload()
        raw = json.loads(next(iter(up.values())).decode())
        user, key = raw['username'], raw['key']
    dst.write_text(json.dumps({'username': user, 'key': key}))
    os.chmod(dst, 0o600)
    print('kaggle auth for', user)

kaggle_auth()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

In [ ]:
from pathlib import Path
DL = Path('/content/dl'); DL.mkdir(exist_ok=True)
SLUGS = ['icold21/kron-llm-code-v18', 'icold21/kron-corpus-2b',
         'icold21/kron-q10-ckpt']
for slug in SLUGS:
    print('[dl]', slug, flush=True)
    subprocess.run([sys.executable, '-m', 'kaggle', 'datasets', 'download',
                    '-d', slug, '-p', str(DL), '--unzip'], check=True)

def find(name):
    hits = sorted(p for p in DL.rglob(name))
    return hits[0] if hits else None

for probe in ('train.bpe.bin', 'val.bpe.bin', 'last.pt', 'kronllm'):
    print(f'{probe:16s}', find(probe))

In [ ]:
WORK = Path('/content/work')
shutil.rmtree(WORK, ignore_errors=True)
for sub in ('data', 'run', 'push'):
    (WORK / sub).mkdir(parents=True)

DATA_FILES = ('train.bpe.bin', 'val.bpe.bin', 'tokenizer.json',
              'tokenizer_meta.json', 'val_index.json')
for name in DATA_FILES:
    src = find(name)
    if src is None:
        print('missing', name)
        continue
    shutil.copyfile(src, WORK / 'data' / name)
    print(f'{name:20s} {src.stat().st_size/1e6:8.1f} MB')

pkg = find('kronllm')
CODE_ROOT = pkg.parent
print('code root', CODE_ROOT)
print('bins:', sorted(p.name for p in (WORK / 'data').iterdir()))

# The corpus is 4.5 GB and the dataset slug does not change between rebuilds, so
# a stale or half-uploaded download looks fine until the loss is wrong.  Read
# the metadata and refuse a corpus that is not the fully-prompted one.
meta = json.loads((WORK / 'data' / 'tokenizer_meta.json').read_text())
dp = meta.get('domain_prompts') or {}
print('phase        ', meta.get('phase'))
print('train tokens ', f"{meta.get('train_tokens', 0):,}")
print('prompted docs', f"{dp.get('prompted_docs', 0):,}", 'added,',
      f"{dp.get('already_prompted_docs', 0):,}", 'already chat, of',
      f"{dp.get('docs', 0):,}", 'at rate', dp.get('rate'))
assert meta.get('phase') == 'epoch', 'wrong bin: this is the decay pass'
assert dp.get('rate') == 1.0, 'corpus predates the full-prompt rebuild'
covered = dp.get('prompted_docs', 0) + dp.get('already_prompted_docs', 0)
assert covered == dp.get('docs'), 'not every document carries a system prompt'
print('corpus is the fully-prompted epoch bin')

In [ ]:
import torch

In [ ]:
MINUTES = 200          # a free T4 is cut near 240; Pro allows 660
CKPT = find('last.pt')
cmd = [sys.executable, '-u', '-m', 'kronllm.cli', 'train', '--exp', 'Q10',
       '--device', 'cuda', '--run-dir', str(WORK / 'run'),
       '--minutes', str(MINUTES)]
if CKPT:
    cmd += ['--resume', str(CKPT)]
    step = torch.load(CKPT, map_location='cpu', weights_only=False).get('step')
    print('resuming from step', step)
else:
    print('no checkpoint found: starting from scratch')
print(' '.join(cmd))
subprocess.run(cmd, cwd=WORK, check=True,
               env=dict(os.environ, PYTHONPATH=str(CODE_ROOT),
                        PYTHONIOENCODING='utf-8'))

In [ ]:
ck = WORK / 'run' / 'last.pt'
if not ck.exists():
    print('no checkpoint written')
else:
    step = torch.load(ck, map_location='cpu', weights_only=False).get('step')
    stage = WORK / 'push'
    shutil.copyfile(ck, stage / 'last.pt')
    # best.pt is weights-only and is what the next session averages with; the
    # soup needs neighbouring checkpoints, so it goes up too.
    best = WORK / 'run' / 'best.pt'
    if best.exists():
        shutil.copyfile(best, stage / 'best.pt')
    (stage / 'dataset-metadata.json').write_text(json.dumps({
        'title': 'Kron Q10 checkpoint', 'id': 'icold21/kron-q10-ckpt',
        'licenses': [{'name': 'other'}]}))
    r = subprocess.run([sys.executable, '-m', 'kaggle', 'datasets', 'version',
                        '-p', str(stage), '-m', f'colab session, step {step}',
                        '--dir-mode', 'zip'])
    print('push exit', r.returncode, 'at step', step)

## Что дальше

- Если `push exit 0` — чекпоинт на Kaggle обновился, следующая сессия (здесь
  или в цепочке) продолжит с этого шага.
- Если push не прошёл, скачай `work/run/last.pt` вручную кнопкой Files слева
  и загрузи как версию датасета `icold21/kron-q10-ckpt`.
- `--minutes` не меняет расписание обучения: это те же шаги, просто сессия
  останавливается сама и пишет чекпоинт, вместо того чтобы быть убитой
  таймаутом Colab.